# Building a tax agent with LangChain + OpenAccountants

This notebook walks through wiring [LangChain](https://github.com/langchain-ai/langchain) up to [OpenAccountants](https://www.openaccountants.com) — an open library of 800+ accountant-verified tax skills across 190+ jurisdictions, served over MCP.

What you get:
- Every tax skill is signed off by a named licensed accountant
- 800+ skills covering income tax, VAT/GST, payroll, formation, cross-border
- The MCP server is public, no API key, no signup
- AGPL-3.0 + attribution (commercial license available)

## Setup

In [ ]:
%pip install langchain langchain-mcp-adapters langchain-anthropic

Set your model API key. This notebook uses Claude but any LangChain-supported chat model works.

In [ ]:
import os
os.environ.setdefault('ANTHROPIC_API_KEY', '<your-key>')

## Connect to the OpenAccountants MCP server

The MCP server lives at `https://www.openaccountants.com/api/mcp`. We use `MultiServerMCPClient` because you'll often want to bundle OpenAccountants with other MCP servers (a QuickBooks MCP, a bank-data MCP, etc.) — the multi-client pattern scales cleanly.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    'openaccountants': {
        'transport': 'streamable_http',
        'url': 'https://www.openaccountants.com/api/mcp',
    }
})

tools = await client.get_tools()
for tool in tools:
    print(f'• {tool.name}: {tool.description[:80]}…')

You should see five tools:

| Tool | Purpose |
|---|---|
| `start` | Front door. Pass an intent + jurisdiction → get a ready-to-execute plan. |
| `list_skills` | List published skills filtered by jurisdiction / category. |
| `get_skill` | Fetch full markdown + provenance footer for one skill. |
| `get_skill_sections` | Same content split into sections (for very large skills). |
| `search_skills` | Keyword search across published skill content. |

## Build the agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    'anthropic:claude-sonnet-4-5',
    tools,
    system_message=(
        'You are a tax assistant. When a user asks a tax or accounting '
        'question, first call `start` to scope the workflow, then '
        '`get_skill` to load the relevant accountant-verified skill, '
        'then walk the user through it. Always cite the skill slug and '
        'the named verifier in the answer. Surface any AUDIT FLASH POINT '
        'markers the skill flags.'
    ),
)

## Ask a tax question

In [ ]:
question = (
    "I'm a Maltese freelancer earning EUR 45,000 from foreign B2B clients. "
    "What VAT obligations do I have, and is there a small-enterprise scheme "
    "I should consider?"
)

response = await agent.ainvoke({'messages': [{'role': 'user', 'content': question}]})
print(response['messages'][-1].content)

The agent should respond with:
1. A call to `start({intent: 'VAT', jurisdiction: 'MT'})` to scope.
2. A call to `get_skill({slug: 'malta-vat-return'})` to load the rules.
3. A walkthrough of Malta VAT Article 10 (standard) vs Article 11 (small enterprise) registration, with the EUR 35,000 threshold, EU place-of-supply reverse charge logic, and the Calendly link to a verified Malta accountant.

## Next steps

- See `examples/02_classify_transactions.py` for a bookkeeping classification example (German Freiberufler).
- See `examples/03_compare_jurisdictions.py` for cross-border tax comparison (MT vs IE vs PT).
- Skills are added continuously; check the [verified accountant network](https://www.openaccountants.com/network) to see who signs off on what.
- For human review of any output, the agent's response always ends with a [Calendly link](https://calendly.com/openaccountants-info/30min) routing to a verified accountant in the relevant jurisdiction.

## Disclaimer

Outputs are working papers, not filed returns. No client-accountant relationship is created by reading or applying these skills. Have a qualified professional review before relying on the output for any tax filing or business decision.